In [1]:
import networkx as nx
import time

In [2]:
cd ..

/home/tuguldur/Development/Research/Dev/ECNDP/Extended-Critical-Node-Detection-Problem/python


In [7]:
def create_custom_graph_with_2_comps(terminal_nodes: list[int]) -> nx.Graph:

  G = nx.Graph()

  C1 = [(1, 2), (2, 3), (3, 4)]
  C2 = [(5, 6), (6, 7)]

  G.add_edges_from(C1)
  G.add_edges_from(C2)

  # Add terminal attribute to nodes
  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False
  
  return G

def create_custom_graph_extreme(terminal_nodes, edge_arrangement):
  
  G = nx.Graph()

  if edge_arrangement == 1:
    C = [(1, 2), (1, 4), (2, 5), (2, 3), (3, 6), (5, 6), (4, 5)]
  if edge_arrangement == 2:
    C = [(1, 2), (1, 3), (2, 5), (2, 4), (3, 5), (5, 6), (4, 6)]
  if edge_arrangement == 3:
    C = [(1, 2), (1, 3), (2, 5), (2, 4), (3, 4), (4, 6), (5, 6)]

  G.add_edges_from(C)

  # Add terminal attribute to nodes
  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False
  
  return G

# 1. New exact IP model

In [10]:
import gurobipy as gp
from gurobipy import GRB
import networkx as nx
from itertools import combinations
import os

LICENSE_PATH = "/home/tuguldur/Development/Research/Dev/ECNDP/Extended-Critical-Node-Detection-Problem/python/notebooks/gurobi.lic"

In [ ]:
def gurobi_status_to_string(status_code):
  status_map = {
    GRB.OPTIMAL: "OPTIMAL",
    GRB.INFEASIBLE: "INFEASIBLE",
    GRB.TIME_LIMIT: "TIME_LIMIT",
    GRB.INTERRUPTED: "INTERRUPTED",
    GRB.SUBOPTIMAL: "SUBOPTIMAL",
    GRB.UNBOUNDED: "UNBOUNDED",
    GRB.INF_OR_UNBD: "INF_OR_UNBD"
  }
  return status_map.get(status_code, f"STATUS_{status_code}")


def build_remaining_graph(graph, deleted_nodes):
  remaining_nodes = [node for node in graph.nodes() if node not in deleted_nodes]
  return graph.subgraph(remaining_nodes).copy()


def extract_solution_data(
  model,
  graph,
  terminals,
  node_removed,
  pairwise_connectivity
):
  solution = {
    "status_code": model.Status,
    "status": gurobi_status_to_string(model.Status),
    "objective": None,
    "deleted_nodes": [],
    "pairwise_connectivity": {},
    "connected_terminal_pairs": [],
    "components_after_deletion": []
  }

  if model.SolCount == 0:
    return solution

  solution["objective"] = model.ObjVal

  deleted_nodes = [
    node for node in graph.nodes()
    if node_removed[node].X > 0.5
  ]
  solution["deleted_nodes"] = deleted_nodes

  terminal_pairs = [(i, j) for i, j in combinations(sorted(terminals), 2)]

  pairwise_connectivity_values = {}
  connected_terminal_pairs = []

  for i, j in terminal_pairs:
    x_value = int(round(pairwise_connectivity[i, j].X))
    pairwise_connectivity_values[(i, j)] = x_value
    if x_value == 1:
      connected_terminal_pairs.append((i, j))

  solution["pairwise_connectivity"] = pairwise_connectivity_values
  solution["connected_terminal_pairs"] = connected_terminal_pairs

  remaining_graph = build_remaining_graph(graph, deleted_nodes)
  components = [sorted(component) for component in nx.connected_components(remaining_graph)]
  components.sort(key=lambda component_nodes: (len(component_nodes), component_nodes))

  solution["components_after_deletion"] = components

  return solution


def solve_ecndp_path_formulation(
  graph,
  terminals,
  budget,
  protect_terminals=False,
  license_path=None,
  time_limit=None,
  mip_gap=None,
  verbose=True
):
  """
  Solve ECNDP with path-based lazy constraints.

  Parameters
  ----------
  graph : networkx.Graph
    Undirected graph.
  terminals : list
    Terminal nodes.
  budget : int
    Maximum number of deleted nodes.
  protect_terminals : bool
    False -> Case 1 (terminals deletable)
    True  -> Case 2 (terminals protected)
  license_path : str or None
    Optional full path to gurobi.lic.
    Must be set before model creation.
  time_limit : float or None
    Time limit in seconds.
  mip_gap : float or None
    Relative MIP gap.
  verbose : bool
    Show Gurobi log or not.
  """

  if license_path is not None:
    os.environ["GRB_LICENSE_FILE"] = license_path

  node_list = list(graph.nodes())
  terminals = sorted(list(terminals))
  terminal_set = set(terminals)

  if not terminal_set.issubset(set(node_list)):
    raise ValueError("All terminals must belong to graph nodes.")

  if budget < 0:
    raise ValueError("Budget must be nonnegative.")

  terminal_pairs = [(i, j) for i, j in combinations(terminals, 2)]

  model = gp.Model("ECNDP_path_lazy")

  if not verbose:
    model.Params.OutputFlag = 0
  if time_limit is not None:
    model.Params.TimeLimit = time_limit
  if mip_gap is not None:
    model.Params.MIPGap = mip_gap

  model.Params.LazyConstraints = 1

  node_removed = model.addVars(
    node_list,
    vtype=GRB.BINARY,
    name="s"
  )

  pairwise_connectivity = model.addVars(
    terminal_pairs,
    vtype=GRB.BINARY,
    name="x"
  )

  model.setObjective(
    gp.quicksum(pairwise_connectivity[i, j] for i, j in terminal_pairs),
    GRB.MINIMIZE
  )

  model.addConstr(
    gp.quicksum(node_removed[node] for node in node_list) <= budget,
    name="budget"
  )

  if protect_terminals:
    for terminal_node in terminals:
      model.addConstr(
        node_removed[terminal_node] == 0,
        name=f"protect_terminal[{terminal_node}]"
      )

  for i, j in terminal_pairs:
    model.addConstr(
      pairwise_connectivity[i, j] <= 1 - node_removed[i],
      name=f"endpoint_i_removed[{i},{j}]"
    )
    model.addConstr(
      pairwise_connectivity[i, j] <= 1 - node_removed[j],
      name=f"endpoint_j_removed[{i},{j}]"
    )

  model._graph = graph
  model._terminals = terminals
  model._terminal_pairs = terminal_pairs
  model._node_removed = node_removed
  model._pairwise_connectivity = pairwise_connectivity

  def lazy_separator(callback_model, where):
    if where != GRB.Callback.MIPSOL:
      return

    graph_data = callback_model._graph
    terminal_pairs_data = callback_model._terminal_pairs
    node_removed_data = callback_model._node_removed
    pairwise_connectivity_data = callback_model._pairwise_connectivity

    node_removed_values = {
      node: callback_model.cbGetSolution(node_removed_data[node])
      for node in graph_data.nodes()
    }

    surviving_nodes = [
      node for node in graph_data.nodes()
      if node_removed_values[node] < 0.5
    ]
    surviving_graph = graph_data.subgraph(surviving_nodes)

    for i, j in terminal_pairs_data:
      x_value = callback_model.cbGetSolution(pairwise_connectivity_data[i, j])

      if x_value > 0.5:
        continue

      if node_removed_values[i] > 0.5 or node_removed_values[j] > 0.5:
        continue

      if nx.has_path(surviving_graph, i, j):
        violating_path = nx.shortest_path(surviving_graph, source=i, target=j)

        callback_model.cbLazy(
          pairwise_connectivity_data[i, j] +
          gp.quicksum(node_removed_data[node] for node in violating_path) >= 1
        )

  model.optimize(lazy_separator)

  solution = extract_solution_data(
    model=model,
    graph=graph,
    terminals=terminals,
    node_removed=node_removed,
    pairwise_connectivity=pairwise_connectivity
  )

  return model, solution


def print_solution(case_name, solution):
  print(f"\n=== {case_name} ===")
  print("Status:", solution["status"])
  print("Objective:", solution["objective"])
  print("Deleted nodes:", solution["deleted_nodes"])
  print("Connected terminal pairs:", solution["connected_terminal_pairs"])

  # print("All pairwise x_ij values:")
  # for pair, value in solution["pairwise_connectivity"].items():
  #   print(f"  x{pair} = {value}")

  print("Connected components after deletion:")
  for component_index, component_nodes in enumerate(solution["components_after_deletion"], start=1):
    print(f"  Component {component_index}: {component_nodes}")

In [6]:
import networkx as nx

graph = nx.Graph()
graph.add_edges_from([
  (1, 2), (2, 3), (3, 4), (4, 5),
  (1, 5), (2, 4), (3, 6), (6, 7), (5, 7)
])

terminals = [1, 3, 5, 7]
budget = 2

# Case 1: terminals may be deleted
model_case_1, solution_case_1 = solve_ecndp_path_formulation(
  graph=graph,
  terminals=terminals,
  budget=budget,
  protect_terminals=False,
  verbose=True
)

print("Case 1")
print(solution_case_1)

# Case 2: terminals are protected
model_case_2, solution_case_2 = solve_ecndp_path_formulation(
  graph=graph,
  terminals=terminals,
  budget=budget,
  protect_terminals=True,
  verbose=True
)

print("Case 2")
print(solution_case_2)

Restricted license - for non-production use only - expires 2027-11-29
Set parameter LazyConstraints to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD Ryzen 9 9950X3D 16-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 13 rows, 13 columns and 31 nonzeros (Min)
Model fingerprint: 0x98d66a67
Model has 6 linear objective coefficients
Variable types: 0 continuous, 13 integer (13 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve time: 0.00s
Presolved: 13 rows, 13 columns, 31 nonzeros
Variable types: 0 continuous, 13 integer (13 binary)

Root relaxation: objective 0.000000e+00, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |  

In [12]:
T = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(T)
G.nodes(data="terminal")

k = 2

terminals = set(T)
graph = G.copy()
budget = int(k)

# Case 1: terminals are deletable
model_case_1, solution_case_1 = solve_ecndp_path_formulation(
    graph=graph,
    terminals=terminals,
    budget=budget,
    protect_terminals=False,
    license_path=LICENSE_PATH,
    verbose=True
  )

# Case 2: terminals are protected
model_case_2, solution_case_2 = solve_ecndp_path_formulation(
  graph=graph,
  terminals=terminals,
  budget=budget,
  protect_terminals=True,
  license_path=LICENSE_PATH,
  verbose=True
  )

print_solution("Case 1 (terminals deletable)", solution_case_1)
print_solution("Case 2 (terminals protected)", solution_case_2)
print("Pairwise connectivity:", solution_case_1["pairwise_connectivity"])

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD Ryzen 9 9950X3D 16-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 21 rows, 17 columns and 47 nonzeros (Min)
Model fingerprint: 0x8a030f3f
Model has 10 linear objective coefficients
Variable types: 0 continuous, 17 integer (17 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve time: 0.00s
Presolved: 21 rows, 17 columns, 47 nonzeros
Variable types: 0 continuous, 17 integer (17 binary)

Root relaxation: objective 0.000000e+00, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf